In [ ]:
from cryptography.hazmat.primitives.asymmetric import ec
import cryptography.hazmat.primitives.hashes as hashes
from cryptography.hazmat.primitives import serialization
import os

In [3]:

# a trusted source ('issuer') generates their own keys
issuer_private_key = ec.generate_private_key(ec.SECP256K1())
issuer_public_key = issuer_private_key.public_key()


# issuer also generates the users keys on their behalf 
user_private_key = ec.generate_private_key(ec.SECP256K1())
user_public_key = user_private_key.public_key()

In [6]:
# A user submits their ID to a trusted source ('issuer')
# The issuer either takes their known personal information, or the submitted photo ID and pulls their information from it

user_info = {
    "name": "John Smith",
    "age": 18,
    "public_key": user_public_key,
    "private_key": user_private_key
}

# the issuer can then determine the result of this boolean
if user_info['age'] >= 18:
  is_18_plus = True
else:
  is_18_plus = False


print("Is user 18+:", is_18_plus)

Is user 18+: True


In [ ]:

# issuer can then use their key to sign an attestation including the users public key and the boolean

# Convert user_public_key to a byte string
user_public_key_bytes = user_public_key.public_bytes(
    encoding=serialization.Encoding.PEM,
    format=serialization.PublicFormat.SubjectPublicKeyInfo
)

# Convert the boolean to a byte string
is_18_plus_bytes = str(is_18_plus).encode('utf-8')

# Combine the data to be signed. Using a separator for clarity.
data_to_sign = user_public_key_bytes + b"|" + is_18_plus_bytes

# signature acts as an attestation signed by the issuer that contains the user public key and their 18+ status
signature = issuer_private_key.sign(
    data_to_sign,
    ec.ECDSA(hashes.SHA256())
)

print("Signature: ", signature)

Signature:  b"0E\x02!\x00\xbe\xa5\x87\xf1\x1a\xf5U\x88\xaa6^K\xf5\x96o-\xe3\x04'2\xe2q\x06\x8a\xf3^\x95x@w\x12\xfc\x02 |\xa9\x03A\x92$\n\x17A\x13U\x18\x8a\x9bY\xf64{\x93\xbfn\xda\x11\xf3Rl\x81\xda'Sf-"


**The next section occurs on the verifier side.**

In [ ]:
# A user submits their signature and the issuer pk to the site
# Verify the signature using the issuer's public key
try:
    issuer_public_key.verify(
        signature,
        data_to_sign,
        ec.ECDSA(hashes.SHA256())
    )
    print("Signature verified.")
except Exception as e:
    print(f"Signature verification failed: {e}")

# Display the original data that was signed
print("\nData that was signed (bytes):", data_to_sign)
print("\nDecoded user public key part:", user_public_key_bytes.decode('utf-8'))
print("Decoded is_18_plus:", is_18_plus_bytes.decode('utf-8'))

# Thus the site can determine that a trusted issuer has verified that a user with user_public_key is either 18 plus or under 18. Determined by decoding is_18_plus

Signature verified.

Data that was signed (bytes): b'-----BEGIN PUBLIC KEY-----\nMFYwEAYHKoZIzj0CAQYFK4EEAAoDQgAEoy1ChiZCNzlJ6ExJ+1SppO+8RVRz8Zmq\nHNeOnlgRZv/pXtTSKSor49f5bm8YXTcX8MAsUXSSL1Hxcve7RYdOlg==\n-----END PUBLIC KEY-----\n|True'

Decoded user public key part: -----BEGIN PUBLIC KEY-----
MFYwEAYHKoZIzj0CAQYFK4EEAAoDQgAEoy1ChiZCNzlJ6ExJ+1SppO+8RVRz8Zmq
HNeOnlgRZv/pXtTSKSor49f5bm8YXTcX8MAsUXSSL1Hxcve7RYdOlg==
-----END PUBLIC KEY-----

Decoded is_18_plus part: True


In [12]:
# in order to prevent someone from signing as you by intercepting your signature and passing it off as theirs
# we use a handshake where the user signs a nonce locally and sends the signature back to the site
# proving that they posess the private key that corresponds with the public key from the issuers attestation.

nonce = os.urandom(16).hex().encode('utf-8')

print(nonce)


signature_runtime = user_private_key.sign(
    nonce,
    ec.ECDSA(hashes.SHA256())
)

try:
    user_public_key.verify(
        signature_runtime,
        nonce,
        ec.ECDSA(hashes.SHA256())
    )
    print("Signature_runtime verification successful. The nonce matches and was signed by the user.")
except Exception as e:
    print(f"Signature_runtime verification failed: {e}. The nonce does not match or was not signed by the user.")

# Display the original nonce for confirmation
print(f"\nOriginal nonce: {nonce.decode('utf-8')}")


b'b09a06a1e001fd67ef317aa35b2bb7a8'
Signature_runtime verification successful. The nonce matches and was signed by the user.

Original nonce: b09a06a1e001fd67ef317aa35b2bb7a8


In [ ]:
# alas we have another issue
# a statement is signed via the issuer public key
# but how does the verifier know that the issuer public key is trusted? it could belong to anyone
# we could create a registry of issuer public keys that adhere to some standard
# but how do sites know what registry to check? how do they know what copy they have? what is most recent? what if we want to revoke trusted status later on?

# we use a solidity smart contract to hold an array of issuer public keys that all sites can check their received signature against via the blockchain 
# this registry can be updated to add or revoke trusted status at any point.
# sites would interact with the smart contract to purely verify that the PK they were given exists as trusted.

# I believe that keeping the entire verification logic as a smart contract on chain is not necessary. 
# ECSDA is purely a mathematical cryptographic verification that can be done easily and cheaply via python

# We must use the same signature algorithm as Ethereum 
